In [1]:
import pandas as pd
df=pd.read_csv('final_normal_data.csv')

In [3]:
df.to_csv('new.csv',index=False)

In [2]:
df2=pd.read_csv('new.csv')

In [3]:
df2.drop('Unnamed: 0',axis=1,inplace=True)

In [4]:
from sklearn.model_selection import train_test_split

X = df2.drop("loan_status", axis=1)
y = df2["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(1076248, 16)
(269062, 16)


In [10]:
sample_df = df.sample(n=250000, random_state=42)

X_sample = sample_df.drop("loan_status", axis=1)
y_sample = sample_df["loan_status"]

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sample,
    y_sample,
    test_size=0.2,
    stratify=y_sample,
    random_state=42
)

In [11]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    tree_method='hist',
    random_state=42
)

param_grid = {
    "n_estimators": [300,400,500],
    "max_depth": [4,5,6,7],
    "learning_rate": [0.03,0.05,0.07],
    "subsample": [0.7,0.8,1],
    "colsample_bytree": [0.7,0.8,1],
    "scale_pos_weight": [2,3,4]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=15,
    scoring='recall',
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_s, y_train_s)

Fitting 3 folds for each of 15 candidates, totalling 45 fits


C:\Users\cheen\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
1 fits failed out of a total of 45.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\cheen\anaconda3\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\cheen\anaconda3\Lib\site-packages\xgboost\core.py", line 751, in inner_f
    return func(**kwargs)
  File "C:\Users\cheen\anaconda3\Lib\site-packages\xgboost\sklearn.py", line 1787, in fit
    train_dmatrix, evals = _wrap_evaluation_matrices(
  

RandomizedSearchCV(cv=3,
                   estimator=XGBClassifier(base_score=None, booster=None,
                                           callbacks=None,
                                           colsample_bylevel=None,
                                           colsample_bynode=None,
                                           colsample_bytree=None, device=None,
                                           early_stopping_rounds=None,
                                           enable_categorical=False,
                                           eval_metric='logloss',
                                           feature_types=None,
                                           feature_weights=None, gamma=None,
                                           grow_policy=None,
                                           importance_type=None,
                                           interaction_cons...
                                           monotone_constraints=None,
                                           multi_strategy=None,
                                           n_estimators=None, n_jobs=None,
                                           num_parallel_tree=None, ...),
                   n_iter=15, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 1],
                                        'learning_rate': [0.03, 0.05, 0.07],
                                        'max_depth': [4, 5, 6, 7],
                                        'n_estimators': [300, 400, 500],
                                        'scale_pos_weight': [2, 3, 4],
                                        'subsample': [0.7, 0.8, 1]},
                   random_state=42, scoring='recall', verbose=2)

In [12]:
print(random_search.best_params_)

{'subsample': 0.8, 'scale_pos_weight': 4, 'n_estimators': 500, 'max_depth': 4, 'learning_rate': 0.07, 'colsample_bytree': 1}


In [13]:
best_xgb = random_search.best_estimator_

best_xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None, colsample_bytree=1,
              device=None, early_stopping_rounds=None, enable_categorical=False,
              eval_metric='logloss', feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.07, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=500, n_jobs=None,
              num_parallel_tree=None, ...)

In [14]:
from sklearn.metrics import classification_report, roc_auc_score

pred = best_xgb.predict(X_test)

print(classification_report(y_test, pred))

probs = best_xgb.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.89      0.64      0.74    215350
           1       0.32      0.68      0.44     53712

    accuracy                           0.65    269062
   macro avg       0.60      0.66      0.59    269062
weighted avg       0.78      0.65      0.68    269062

ROC-AUC: 0.7198250840641615


In [15]:
import numpy as np
from sklearn.metrics import classification_report

probs = best_xgb.predict_proba(X_test)[:,1]

# Try different thresholds
for t in [0.5, 0.45, 0.4, 0.35, 0.3]:

    print(f"\nThreshold = {t}")
    preds = (probs >= t).astype(int)
    print(classification_report(y_test, preds))


Threshold = 0.5
              precision    recall  f1-score   support

           0       0.89      0.64      0.74    215350
           1       0.32      0.68      0.44     53712

    accuracy                           0.65    269062
   macro avg       0.60      0.66      0.59    269062
weighted avg       0.78      0.65      0.68    269062


Threshold = 0.45
              precision    recall  f1-score   support

           0       0.90      0.55      0.68    215350
           1       0.30      0.76      0.43     53712

    accuracy                           0.59    269062
   macro avg       0.60      0.66      0.55    269062
weighted avg       0.78      0.59      0.63    269062


Threshold = 0.4
              precision    recall  f1-score   support

           0       0.92      0.45      0.61    215350
           1       0.28      0.83      0.41     53712

    accuracy                           0.53    269062
   macro avg       0.60      0.64      0.51    269062
weighted avg       0.7

In [5]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.6, random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [10]:
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    use_label_encoder=False,
    random_state=42
)

param_grid = {
    'n_estimators': [300, 400, 500],
    'max_depth': [3, 4, 5],
    'learning_rate': [0.05, 0.07, 0.1],
    'subsample': [0.8, 1],
    'colsample_bytree': [0.8, 1]
}

random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=15,
    scoring='recall',  # ← Change f1 to recall!
    cv=3,
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train_sm, y_train_sm)

best_xgb = random_search.best_estimator_

C:\Users\cheen\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:27:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [11]:
 best_xgb.fit(X_train_sm, y_train_sm)

C:\Users\cheen\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:28:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=4, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=None,
              num_parallel_tree=None, ...)

In [12]:
from sklearn.metrics import classification_report, roc_auc_score

pred = best_xgb.predict(X_test)

print(classification_report(y_test, pred))

probs = best_xgb.predict_proba(X_test)[:,1]
print("ROC-AUC:", roc_auc_score(y_test, probs))

              precision    recall  f1-score   support

           0       0.81      0.98      0.89    215350
           1       0.53      0.09      0.15     53712

    accuracy                           0.80    269062
   macro avg       0.67      0.53      0.52    269062
weighted avg       0.76      0.80      0.74    269062

ROC-AUC: 0.7130710904286094
